In [1]:
print (123)

123


# Loading the RAG answers


In [2]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [3]:
# We'll compare the RAG answer with the original answer from the FAQ. This checks if the RAG pipeline is producing answers that match the ground truth.

# First, define the output format:

from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [4]:
# The judge returns two fields. 
# The score gives us a metric we can aggregate. 
# The reasoning explains the score, which helps when we look at bad examples.
# First, write the judge instructions. This tells the judge what to compare and how to assign the score.

In [5]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [6]:
# Then define the prompt template. This is the data we pass to the judge for each answer.

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [7]:
# Import the structured-output helper:

from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [22]:
answers

[{'question': 'Is it okay to join the course late if I just found it now?',
  'answer_llm': 'Yes. You can still join the course late. If you want a certificate, make sure to submit your project while submissions are still open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'answer_llm': 'Yes, you can still take the course even if you missed the start date. You can start whenever you want, and the course materials are available. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still eligible for 

In [8]:
# Take one record:
rec = answers[0]
rec

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes. You can still join the course late. If you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [9]:
# Create the judge prompt:

prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)


In [23]:
prompt

'Question:\nIs it okay to join the course late if I just found it now?\n\nOriginal Answer (ground truth):\nYes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nAI Answer:\nYes. You can still join the course late. If you want a certificate, make sure to submit your project while submissions are still open.'

In [24]:
print(prompt)

Question:
Is it okay to join the course late if I just found it now?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
Yes. You can still join the course late. If you want a certificate, make sure to submit your project while submissions are still open.


In [25]:
# Call the judge
# Send instruction, prompt, output type to the LLM

eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key points: late joining is allowed, and certificate eligibility depends on submitting the project while submissions are open. This is semantically equivalent to the ground truth.', score='good')

In [11]:
# Check the cost
calc_price(usage)

{'input_cost': 0.0002205, 'output_cost': 0.0002295, 'total_cost': 0.00045}

In [26]:
# Now put the same logic into a function - everything that has been implemented so far

def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [27]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late joining is allowed, and certificate eligibility depends on submitting the project before submissions close. It is semantically equivalent.', score='good')

# Running the judge


In [14]:
# Run the evaluation for all answers

def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [15]:
# Use the same parallel processing helper:

from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/315 [00:00<?, ?it/s]

In [16]:
# Split the results

evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [17]:
df_eval = pd.DataFrame(evaluations)

In [18]:
calc_total_price(usages)

0.20654699999999998

In [28]:
df_eval

,question,document,score,reasoning
0,Is it okay to join the course late if I just f...,74eb249bbf,good,The AI answer preserves the key meaning of the...
1,Can I still take this course even if I missed ...,74eb249bbf,good,The AI answer matches the ground truth: it say...
2,If I join after the course has already started...,74eb249bbf,bad,The AI answer does not convey the ground truth...
3,Do I need to submit my project before submissi...,74eb249bbf,good,The AI answer preserves the key point from the...
4,I’m a bit late to the course—what do I need to...,74eb249bbf,bad,The ground truth says the only requirement men...
...,...,...,...,...
310,Why do I get a 401 Client Error when using the...,4b30b918bc,good,The AI answer captures the key idea that a 401...
311,What's the easiest way to force-install reques...,4b30b918bc,good,The AI answer gives the same installation comm...
312,Can I install requests straight from the GitHu...,4b30b918bc,good,The AI answer provides the same installation c...
313,"If pip keeps pulling requests v2.28, what exac...",4b30b918bc,good,The AI answer provides the exact install comma...


In [19]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 282/315 = 89.52%


In [20]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
2,If I join after the course has already started...,74eb249bbf,bad,The AI answer does not convey the ground truth...
4,I’m a bit late to the course—what do I need to...,74eb249bbf,bad,The ground truth says the only requirement men...
27,What do I actually need to pass in order to ge...,9f689c185f,bad,The ground truth says the only requirement for...
33,How will I know when a module is actually read...,96286b4be4,bad,The AI answer fails to provide the key criteri...
39,Can you tell me when the course is coming back?,bd31146b0e,bad,The ground truth gives a specific return time:...
